In [1]:
# ==============================
# WHL PHASE 1A - TEAM ANALYSIS
# ==============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# Load season data
games = pd.read_excel("whl_2025.xlsx")

# Load matchup file (Round 1)
matchups = pd.read_excel("WHSDSC_Rnd1_matchups.xlsx")

# Optional reference only
data_dict = pd.read_excel("WHSDSC_2026_DataDictionary.xlsx")

print("Games shape:", games.shape)
games.head()

Games shape: (25827, 26)


,game_id,record_id,home_team,away_team,went_ot,home_off_line,home_def_pairing,away_off_line,away_def_pairing,home_goalie,...,home_goals,away_assists,away_shots,away_xg,away_max_xg,away_goals,home_penalties_committed,home_penalty_minutes,away_penalties_committed,away_penalty_minutes
0,game_1,record_1,thailand,pakistan,0,PP_kill_dwn,PP_kill_dwn,PP_up,PP_up,player_id_142,...,0,2,9,1.4645,0.2166,1,7,14,1,2
1,game_1,record_2,thailand,pakistan,0,second_off,second_def,second_off,second_def,player_id_142,...,0,2,1,0.0928,0.0928,1,0,0,0,0
2,game_1,record_3,thailand,pakistan,0,first_off,second_def,second_off,second_def,player_id_142,...,0,0,2,0.1880,0.0940,0,0,0,0,0
3,game_1,record_4,thailand,pakistan,0,second_off,first_def,second_off,first_def,player_id_142,...,0,0,1,0.0727,0.0727,0,0,0,0,0
4,game_1,record_5,thailand,pakistan,0,second_off,second_def,first_off,second_def,player_id_142,...,0,2,1,0.0769,0.0769,1,0,0,0,0


In [10]:
HOME_TEAM = "home_team"
AWAY_TEAM = "away_team"
HOME_GOALS = "home_goals"
AWAY_GOALS = "away_goals"
print(games.columns)

Index(['game_id', 'record_id', 'home_team', 'away_team', 'went_ot',
       'home_off_line', 'home_def_pairing', 'away_off_line',
       'away_def_pairing', 'home_goalie', 'away_goalie', 'toi', 'home_assists',
       'home_shots', 'home_xg', 'home_max_xg', 'home_goals', 'away_assists',
       'away_shots', 'away_xg', 'away_max_xg', 'away_goals',
       'home_penalties_committed', 'home_penalty_minutes',
       'away_penalties_committed', 'away_penalty_minutes'],
      dtype='object')


In [11]:
# Determine wins
games["home_win"] = (games[HOME_GOALS] > games[AWAY_GOALS]).astype(int)
games["away_win"] = (games[AWAY_GOALS] > games[HOME_GOALS]).astype(int)

# Points system
games["home_pts"] = np.where(games["home_win"] == 1, 2, 0)
games["away_pts"] = np.where(games["away_win"] == 1, 2, 0)

# Convert to long format
home_rows = pd.DataFrame({
    "Team": games[HOME_TEAM],
    "Opponent": games[AWAY_TEAM],
    "GF": games[HOME_GOALS],
    "GA": games[AWAY_GOALS],
    "Win": games["home_win"],
    "Points": games["home_pts"],
    "Home": 1
})

away_rows = pd.DataFrame({
    "Team": games[AWAY_TEAM],
    "Opponent": games[HOME_TEAM],
    "GF": games[AWAY_GOALS],
    "GA": games[HOME_GOALS],
    "Win": games["away_win"],
    "Points": games["away_pts"],
    "Home": 0
})

long = pd.concat([home_rows, away_rows], ignore_index=True)
long["GD"] = long["GF"] - long["GA"]

# League table
league_table = (
    long.groupby("Team")
    .agg(
        GP=("Team","size"),
        Wins=("Win","sum"),
        GF=("GF","sum"),
        GA=("GA","sum"),
        GD=("GD","sum"),
        Points=("Points","sum")
    )
    .reset_index()
)

league_table["Points%"] = league_table["Points"] / (2 * league_table["GP"])

league_table = league_table.sort_values(
    ["Points","GD","GF"], ascending=False
).reset_index(drop=True)

league_table.insert(0, "Rank", league_table.index + 1)

league_table.head(32)

,Rank,Team,GP,Wins,GF,GA,GD,Points,Points%
0,1,pakistan,1602,201,263,212,51,402,0.125468
1,2,thailand,1642,200,294,248,46,400,0.121803
2,3,brazil,1611,194,276,189,87,388,0.120422
3,4,ethiopia,1613,193,267,247,20,386,0.119653
4,5,serbia,1622,188,269,264,5,376,0.115906
5,6,south_korea,1604,188,288,302,-14,376,0.117207
6,7,peru,1613,179,256,178,78,358,0.110973
7,8,panama,1616,179,255,213,42,358,0.110767
8,9,uk,1631,179,244,211,33,358,0.109749
9,10,iceland,1642,174,238,209,29,348,0.105968


In [12]:
teams = pd.unique(pd.concat([games[HOME_TEAM], games[AWAY_TEAM]]))
elo = {t: 1500 for t in teams}

HOME_ADV = 60
K = 20

def expected(a,b):
    return 1 / (1 + 10 ** ((b-a)/400))

# Elo updates
for _, row in games.iterrows():
    h = row[HOME_TEAM]
    a = row[AWAY_TEAM]
    hg = row[HOME_GOALS]
    ag = row[AWAY_GOALS]

    rh = elo[h] + HOME_ADV
    ra = elo[a]

    exp_h = expected(rh, ra)

    score_h = 1 if hg > ag else 0

    elo[h] += K * (score_h - exp_h)
    elo[a] += K * ((1-score_h) - (1-exp_h))

elo_df = pd.DataFrame({
    "Team": list(elo.keys()),
    "ELO": list(elo.values())
})

In [13]:
power = league_table.merge(elo_df, on="Team")

power["GD_per_game"] = power["GD"] / power["GP"]

def z(x):
    return (x - x.mean()) / x.std()

power["z_pts"] = z(power["Points%"])
power["z_gd"] = z(power["GD_per_game"])
power["z_elo"] = z(power["ELO"])

# Weighted power score
power["PowerScore"] = (
    0.40*power["z_elo"] +
    0.35*power["z_gd"] +
    0.25*power["z_pts"]
)

power = power.sort_values("PowerScore", ascending=False).reset_index(drop=True)
power.insert(0, "PowerRank", power.index+1)

power[["PowerRank","Team","PowerScore"]]

,PowerRank,Team,PowerScore
0,1,brazil,1.837640
1,2,peru,1.238860
2,3,thailand,1.087468
3,4,panama,0.951711
4,5,ethiopia,0.838617
5,6,pakistan,0.821612
6,7,indonesia,0.760830
7,8,serbia,0.568653
8,9,south_korea,0.337317
9,10,netherlands,0.326198


In [15]:
def win_prob(home, away):
    rh = elo[home] + HOME_ADV
    ra = elo[away]
    return expected(rh, ra)

matchups["HomeWinProb"] = matchups.apply(
    lambda r: win_prob(r[HOME_TEAM], r[AWAY_TEAM]),
    axis=1
)

matchups[["home_team","away_team","HomeWinProb"]]

,home_team,away_team,HomeWinProb
0,brazil,kazakhstan,0.958802
1,netherlands,mongolia,0.353208
2,peru,rwanda,0.702003
3,thailand,oman,0.905609
4,pakistan,germany,0.551235
5,india,usa,0.773207
6,panama,switzerland,0.695584
7,iceland,canada,0.694766
8,china,france,0.791835
9,philippines,morocco,0.288247


In [16]:
matchups.to_csv("phase1a_predictions.csv", index=False)
print("Submission file created.")

Submission file created.
